# BiTro Bulk Pipeline (Notebook)

This notebook provides a configurable bulk pipeline for BiTro: feature extraction, clustering labels, graph construction, and training.
Edit the config cell first, then run cell by cell.

In [ ]:
import os
import subprocess
from pathlib import Path

def run_cmd(cmd, env=None, cwd=None):
    print(f"\n[RUN] {' '.join(cmd)}")
    result = subprocess.run(cmd, env=env, cwd=cwd, text=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


In [ ]:
# =========================
# Editable configuration
# =========================
PROJECT_ROOT = Path('.').resolve()

# Feature extraction inputs (set these to your train/test data roots)
TRAIN_IMAGE_FOLDER = Path('/mnt/elements2/PRAD_train')
TRAIN_JSON_FOLDER = Path('/mnt/elements1/PRAD_train')
TRAIN_MAT_FOLDER = Path('/mnt/elements1/PRAD_train')

TEST_IMAGE_FOLDER = Path('/mnt/elements2/PRAD_test')
TEST_JSON_FOLDER = Path('/mnt/elements1/PRAD_test')
TEST_MAT_FOLDER = Path('/mnt/elements1/PRAD_test')

TRAIN_FEATURE_DIR = Path('/mnt/elements/ouput_features/PRAD_train')
TEST_FEATURE_DIR = Path('/mnt/elements/ouput_features/PRAD_test')

# Bulk expression and patch/WSI inputs for graph construction
BULK_CSV_PATH = Path('/mnt/elements/PRAD/tpm-TCGA-PRAD.csv')
PATCHES_DIR = Path('/mnt/elements/PRAD/PRAD')
WSI_INPUT_DIR = Path('/mnt/elements/PRAD/PRAD_svs')

GRAPH_OUTPUT_DIR = Path('/mnt/elements/PRAD/bulk_PRAD_graphs_new_all_graph')
CHECKPOINT_DIR = Path('/mnt/elements/PRAD/bulk_PRAD_graphs_checkpoints')

# KMeans cluster label settings for parquet files
N_CLUSTERS = 7
CLUSTER_RANDOM_STATE = 42

# Feature extraction settings
PCA_COMPONENTS = 128
CHUNK_SIZE = 100

# Graph settings
INTRA_PATCH_DISTANCE_THRESHOLD = 256
INTER_PATCH_K = 8
MAX_CELLS_PER_PATCH = None
MAX_TRAIN_SLIDES = 200
MAX_TEST_SLIDES = 50

# Training settings
GENE_LIST_FILE = Path('/root/autodl-tmp/PRAD_intersection_genes.txt')
FEATURES_TSV_FILE = Path('/root/autodl-tmp/features.tsv')
TPM_CSV_FILE = Path('/root/autodl-tmp/tpm-TCGA-PRAD-intersect-normalized.csv')

BATCH_SIZE = 2
GRAPH_BATCH_SIZE = 128
NUM_EPOCHS = 60
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

USE_LORA = True
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
required_paths = [
    PROJECT_ROOT / 'utils' / 'extract_bulk_features_dinov3.py',
    PROJECT_ROOT / 'utils' / 'bulk_graph_construction.py',
    PROJECT_ROOT / 'bulk_model' / 'train.py',
]
for p in required_paths:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

for d in [TRAIN_FEATURE_DIR, TEST_FEATURE_DIR, GRAPH_OUTPUT_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Config check done')

In [ ]:
# Step 1: feature extraction for train split
from utils.extract_bulk_features_dinov3 import batch_extract_features

batch_extract_features(
    image_folder=str(TRAIN_IMAGE_FOLDER),
    json_folder=str(TRAIN_JSON_FOLDER),
    mat_folder=str(TRAIN_MAT_FOLDER),
    output_folder=str(TRAIN_FEATURE_DIR),
    model_name='dinov3',
    pca_components=PCA_COMPONENTS,
    chunk_size=CHUNK_SIZE,
)

In [ ]:
# Step 2: feature extraction for test split
from utils.extract_bulk_features_dinov3 import batch_extract_features

batch_extract_features(
    image_folder=str(TEST_IMAGE_FOLDER),
    json_folder=str(TEST_JSON_FOLDER),
    mat_folder=str(TEST_MAT_FOLDER),
    output_folder=str(TEST_FEATURE_DIR),
    model_name='dinov3',
    pca_components=PCA_COMPONENTS,
    chunk_size=CHUNK_SIZE,
)

In [ ]:
# Step 3: add cluster_label column to each parquet (required by bulk_graph_construction.py)
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans

def append_cluster_labels_to_parquet(feature_dir, n_clusters=7, random_state=42):
    feature_dir = Path(feature_dir)
    files = sorted(feature_dir.glob('*.parquet'))
    print(f'Found {len(files)} parquet files in {feature_dir}')

    for fp in files:
        df = pd.read_parquet(fp)
        feature_cols = sorted([c for c in df.columns if c.startswith('feature_')], key=lambda x: int(x.split('_')[1]))
        if len(feature_cols) == 0:
            print(f'[SKIP] no feature_* columns: {fp.name}')
            continue

        if 'cluster_label' in df.columns and len(df['cluster_label']) == len(df):
            print(f'[SKIP] already has cluster_label: {fp.name}')
            continue

        x = df[feature_cols].to_numpy(dtype=np.float32, copy=False)
        if x.shape[0] == 0:
            print(f'[SKIP] empty parquet: {fp.name}')
            continue

        k = min(n_clusters, x.shape[0])
        if k <= 1:
            labels = np.zeros((x.shape[0],), dtype=np.int32)
        else:
            km = MiniBatchKMeans(n_clusters=k, random_state=random_state, batch_size=4096)
            labels = km.fit_predict(x).astype(np.int32)

        df['cluster_label'] = labels
        df.to_parquet(fp, engine='pyarrow', compression='snappy', index=False)
        print(f'[OK] wrote cluster_label: {fp.name}')

append_cluster_labels_to_parquet(TRAIN_FEATURE_DIR, n_clusters=N_CLUSTERS, random_state=CLUSTER_RANDOM_STATE)
append_cluster_labels_to_parquet(TEST_FEATURE_DIR, n_clusters=N_CLUSTERS, random_state=CLUSTER_RANDOM_STATE)

In [ ]:
# Step 4: construct bulk graphs
from utils.bulk_graph_construction import BulkStaticGraphBuilder

builder = BulkStaticGraphBuilder(
    train_features_dir=str(TRAIN_FEATURE_DIR),
    test_features_dir=str(TEST_FEATURE_DIR),
    bulk_csv_path=str(BULK_CSV_PATH),
    patches_dir=str(PATCHES_DIR),
    wsi_input_dir=str(WSI_INPUT_DIR),
    intra_patch_distance_threshold=INTRA_PATCH_DISTANCE_THRESHOLD,
    inter_patch_k_neighbors=INTER_PATCH_K,
    use_deep_features=True,
    feature_dim=128,
    max_cells_per_patch=MAX_CELLS_PER_PATCH,
    max_train_slides=MAX_TRAIN_SLIDES,
    max_test_slides=MAX_TEST_SLIDES,
    checkpoint_dir=str(CHECKPOINT_DIR),
)

builder.load_bulk_data()
builder.process_all_slides_new_logic()
metadata = builder.save_graphs_slide_logic(str(GRAPH_OUTPUT_DIR))
builder.save_selected_feature_filenames(str(GRAPH_OUTPUT_DIR))
print('Done graph construction')

In [ ]:
# Step 5: train bulk model
cmd = [
    'python', 'bulk_model/train.py',
    '--graph-data-dir', str(GRAPH_OUTPUT_DIR),
    '--gene-list-file', str(GENE_LIST_FILE),
    '--features-file', str(FEATURES_TSV_FILE),
    '--tpm-csv-file', str(TPM_CSV_FILE),
    '--batch-size', str(BATCH_SIZE),
    '--graph-batch-size', str(GRAPH_BATCH_SIZE),
    '--num-epochs', str(NUM_EPOCHS),
    '--learning-rate', str(LEARNING_RATE),
    '--weight-decay', str(WEIGHT_DECAY),
]

if USE_LORA:
    cmd += [
        '--use-lora',
        '--lora-r', str(LORA_R),
        '--lora-alpha', str(LORA_ALPHA),
        '--lora-dropout', str(LORA_DROPOUT),
        '--lora-freeze-base',
    ]

run_cmd(cmd, cwd=str(PROJECT_ROOT))

In [ ]:
# Step 6: quick artifact check
print('Train feature dir:', TRAIN_FEATURE_DIR)
print('Test feature dir:', TEST_FEATURE_DIR)
print('Graph output dir:', GRAPH_OUTPUT_DIR)

if GRAPH_OUTPUT_DIR.exists():
    for p in sorted(GRAPH_OUTPUT_DIR.glob('*')):
        print(' -', p.name)